# From delivered tables to the adaptation corpus

Explain the inputs used by training, the held-out dataset design and the exposure implications of each sampling protocol. These are local corpus measurements, separate from the downloaded training outcomes.

The corpus defines the population to which the CPT study applies; related source populations and local preprocessing limit broader generalization.


## 0. Evidence and scope

The availability tables distinguish observed measurements from unavailable outcomes. Missing measurements are not zero effects.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, publication as pub, diagnostics as dg
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
from src.data.dataset_names import display_frame
style.apply()
sink = FigureSaver('general/02_processed_data')
report = cp.NotebookReport('From delivered tables to the adaptation corpus', sink=sink)
from src.visualize import corpus as cv
print(source_description())
report.add('0. Evidence and scope', source_description() + '\n' + 'The corpus defines the population to which the CPT study applies; related source populations and local preprocessing limit broader generalization.')

## 1. Processed inventory

Report available tables and predictor missingness after the registered cleaning rules.

In [ ]:
data = cv.processed_inventory()
report.table('Processed tables', data[['track','dataset_id','rows','features','missing']])
report.add('1. Processed inventory', cv.inventory_summary(data, 'Processed'))


## 2. Preprocessing footprint

Registration already contains dataset-specific corrections. The diagonal comparison is against that registered shape, not necessarily against the untouched download.

In [ ]:
cp.show(sink, cv.plot_cleaning(data))
report.table('Registered and processed shapes', data[['track','dataset_id','raw_rows','post_rows','raw_features','post_features']])
report.add('2. Preprocessing footprint', 'Differences are descriptive; discarded rows are not automatically evidence of poor data quality.')


## 3. Training-input geometry and quality

Separate broad geometry from readable per-dataset pages.

In [ ]:
cp.show(sink, cv.plot_geometry(data, 'Processed'))
cp.show(sink, cv.plot_profiles(data, 'Processed'))
report.add('3. Training-input geometry and quality', 'Predictor missingness excludes the target.')


## 4. Target distributions

PD imbalance and LGD shape motivate different training objectives and evaluation metrics. LGD distributions reflect the registered preprocessing, including its clipping rules.

In [ ]:
cp.show(sink, cv.plot_target_profiles(data))
cp.show(sink, cv.plot_lgd_distributions())
report.table('Target descriptors', data[['track','dataset_id','minority_class_ratio','target_mean','target_std']])
report.add('4. Target distributions', 'Histograms describe the processed targets; no additional clipping is applied by the plot.')


## 5. Exact held-out dataset assignments

The four dataset partitions are distinct from the five row-level outer folds later used for final evaluation. Every dataset is held out from adaptation once.

In [ ]:
partitions = cv.partition_table()
report.table('Dataset partition membership', partitions)
cp.show(sink, cv.plot_partitions(partitions))
report.add('5. Exact held-out dataset assignments', partitions.groupby(['track','fold']).size().rename('held_out_tables').to_string())


## 6. Sampling and exposure illustration

Show why full traversal, one-sample updates and gradient accumulation do not have equal data or compute budgets even when their optimizer-update count is equal.

The common marker gives equal optimizer-step share for one sample and accumulation; their row exposure differs. The multiplier beside each dataset gives its full-pass step share relative to that equal-table share.

In [ ]:
exposure = cv.planned_exposure(data)
cp.show(sink, cv.plot_exposure(exposure))
report.table('Illustrative exposure shares', exposure[['track','dataset_id','rows','chunks','row_share','one_sample_step_share','full_pass_step_share','accumulate_step_share','full_pass_share_ratio']])
report.add('6. Sampling and exposure illustration', 'Illustration only: a 10,000-row cap, all processed tables and no skipped updates. Actual runs use base-specific caps and training partitions.')


## 7. Quality flags and interpretation

Flags are descriptive prompts for inspection, not unrecorded exclusion rules.

In [ ]:
from src.data.exploration import find_anomalous_datasets
anomalies = display_frame(find_anomalous_datasets())
report.table('Quality flags', anomalies)
report.add('7. Quality flags and interpretation', 'No configured anomaly flags.' if anomalies.empty else 'Flagged datasets remain in the registered corpus unless an explicit design change is made.')


## Complete text summary

Complete numerical values and figure captions, following the section order above.

In [ ]:
print(report.summary(sink))
